In [2]:
import pandas as pd
import numpy as np

# ==========================
# LOAD DATA
# ==========================
df = pd.read_csv("cleaned_output.csv")

print("Original Shape:", df.shape)

# ==========================
# COLUMN CLEANUP
# ==========================
df.columns = df.columns.str.strip()

# ==========================
# AGE CLEANING
# ==========================
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

df.loc[df["Age"] < 0, "Age"] = np.nan
df.loc[df["Age"] > 120, "Age"] = np.nan

# ==========================
# INFECTION FREQUENCY
# ==========================
df["Infection_Freq"] = pd.to_numeric(
    df["Infection_Freq"],
    errors="coerce"
)

df.loc[df["Infection_Freq"] < 0, "Infection_Freq"] = np.nan

# ==========================
# GENDER CLEANING
# ==========================
df["Gender"] = (
    df["Gender"]
    .astype(str)
    .str.strip()
    .str.upper()
)

gender_map = {
    "MALE": "M",
    "M": "M",
    "FEMALE": "F",
    "F": "F"
}

df["Gender"] = df["Gender"].replace(gender_map)

# ==========================
# YES / NO COLUMNS
# ==========================
binary_cols = [
    "Diabetes",
    "Hypertension",
    "Hospital_before"
]

for col in binary_cols:

    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    mapping = {
        "yes": "Yes",
        "y": "Yes",
        "true": "Yes",
        "1": "Yes",

        "no": "No",
        "n": "No",
        "false": "No",
        "0": "No"
    }

    df[col] = df[col].replace(mapping)

# ==========================
# STRAIN CLEANING
# ==========================
df["Souches"] = (
    df["Souches"]
    .astype(str)
    .str.strip()
)

# common typo corrections
strain_map = {

    "Klbsiella pneumoniae":
        "Klebsiella pneumoniae",

    "Klebsie.lla pneumoniae":
        "Klebsiella pneumoniae",

    "Enter.bacteria spp.":
        "Enterobacteria spp.",

    "Enteobacteria spp.":
        "Enterobacteria spp.",

    "?":
        "Unknown",

    "missing":
        "Unknown",

    "nan":
        "Unknown"
}

df["Souches"] = df["Souches"].replace(strain_map)

# ==========================
# GROUP RARE STRAINS
# ==========================
strain_counts = df["Souches"].value_counts()

rare_strains = strain_counts[
    strain_counts < 20
].index

df["Souches"] = df["Souches"].replace(
    rare_strains,
    "Other"
)

# ==========================
# TARGET CLEANING
# ==========================
targets = [
    'AMX/AMP',
    'AMC',
    'CZ',
    'FOX',
    'CTX/CRO',
    'IPM',
    'GEN',
    'AN',
    'Acide nalidixique',
    'ofx',
    'CIP',
    'C',
    'Co-trimoxazole',
    'Furanes',
    'colistine'
]

for col in targets:

    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # Resistant
    df.loc[
        df[col].isin(["r"]),
        col
    ] = 1

    # Susceptible
    df.loc[
        df[col].isin(["s"]),
        col
    ] = 0

    # Unknown values
    df.loc[
        df[col].isin([
            "intermediate",
            "i",
            "?",
            "missing",
            "nan",
            ""
        ]),
        col
    ] = np.nan

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

# ==========================
# FEATURE IMPUTATION
# ==========================
df["Age"] = df["Age"].fillna(
    df["Age"].median()
)

df["Infection_Freq"] = df[
    "Infection_Freq"
].fillna(
    df["Infection_Freq"].median()
)

for col in [
    "Gender",
    "Diabetes",
    "Hypertension",
    "Hospital_before",
    "Souches"
]:
    df[col] = df[col].fillna(
        df[col].mode()[0]
    )

# ==========================
# REMOVE ROWS WITH
# ALL TARGETS MISSING
# ==========================
df = df.dropna(
    subset=targets,
    how="all"
)

print("Final Shape:", df.shape)

# ==========================
# SAVE CLEAN DATA
# ==========================
df.to_csv(
    "cleaned_output_v2.csv",
    index=False
)

print("\nSaved as cleaned_output_v2.csv")

Original Shape: (10710, 23)
Final Shape: (9957, 23)

Saved as cleaned_output_v2.csv


In [3]:
targets = [
    'AMX/AMP','AMC','CZ','FOX','CTX/CRO',
    'IPM','GEN','AN','Acide nalidixique',
    'ofx','CIP','C','Co-trimoxazole',
    'Furanes','colistine'
]

for col in targets:

    temp = df[df[col].notna()]

    resistant = (temp[col] == 1).sum()
    susceptible = (temp[col] == 0).sum()

    total = resistant + susceptible

    print(
        f"{col:20s}"
        f" Resistant={resistant:5d}"
        f" ({100*resistant/total:.2f}%)"
        f" Susceptible={susceptible:5d}"
        f" ({100*susceptible/total:.2f}%)"
    )

AMX/AMP              Resistant= 5712 (58.45%) Susceptible= 4061 (41.55%)
AMC                  Resistant= 5815 (59.43%) Susceptible= 3970 (40.57%)
CZ                   Resistant= 5690 (58.14%) Susceptible= 4097 (41.86%)
FOX                  Resistant= 5735 (58.62%) Susceptible= 4049 (41.38%)
CTX/CRO              Resistant= 5748 (58.81%) Susceptible= 4026 (41.19%)
IPM                  Resistant= 5717 (58.51%) Susceptible= 4054 (41.49%)
GEN                  Resistant= 1936 (19.78%) Susceptible= 7854 (80.22%)
AN                   Resistant= 1905 (19.48%) Susceptible= 7875 (80.52%)
Acide nalidixique    Resistant= 1388 (14.21%) Susceptible= 8379 (85.79%)
ofx                  Resistant= 1381 (14.11%) Susceptible= 8406 (85.89%)
CIP                  Resistant= 1447 (14.80%) Susceptible= 8331 (85.20%)
C                    Resistant= 1385 (14.17%) Susceptible= 8390 (85.83%)
Co-trimoxazole       Resistant= 1413 (14.44%) Susceptible= 8370 (85.56%)
Furanes              Resistant= 1326 (13.55%) Susce

In [4]:
df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[0,18,40,60,120],
    labels=[
        "Child",
        "Young_Adult",
        "Adult",
        "Senior"
    ]
)

In [5]:
df["Comorbidity_Score"] = (
    (df["Diabetes"] == "Yes").astype(int)
    +
    (df["Hypertension"] == "Yes").astype(int)
)

In [6]:
df["Hospital_Risk"] = (
    df["Hospital_before"] == "Yes"
).astype(int)

In [7]:
df["High_Risk_Patient"] = (
    (
        (df["Hospital_before"] == "Yes")
        &
        (df["Infection_Freq"] >= 3)
    )
).astype(int)

In [8]:
strain_freq = df["Souches"].value_counts()

df["Strain_Frequency"] = (
    df["Souches"]
    .map(strain_freq)
)

In [9]:
df["Risk_Score"] = (
    (df["Diabetes"] == "Yes").astype(int)
    +
    (df["Hypertension"] == "Yes").astype(int)
    +
    (df["Hospital_before"] == "Yes").astype(int)
    +
    (df["Infection_Freq"] >= 3).astype(int)
)